In [1]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.model_selection import GroupKFold
from sklearn.metrics import log_loss

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("[warning] xgboost not installed — skipping XGBoost.")

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
TRAIN_PATH = "/Users/ak/Documents/UNSW Classes/DATA5925/CODE/Train and test/train_split.csv"
TEST_PATH  = "/Users/ak/Documents/UNSW Classes/DATA5925/CODE/Train and test/test_split.csv"
OUTPUT_DIR = "/Users/ak/Documents/UNSW Classes/DATA5925/CODE/single vs multi species/outputs"

TARGET = "pres.abs"
SPECIES_COL = "Species"
PLOT_COL = "plot"
NUMERIC_FEATURES = ["easting", "northing", "disturb", "rainann",
                     "soildepth", "soilfert", "tempann", "topo"]
CLIP = 1e-6

N_SPLITS_RANDOM = 5
N_REPEATS_RANDOM = 3      # total random-CV fold evaluations = 5 x 3 = 15
N_BLOCKS_SPATIAL = 6
N_REPEATS_SPATIAL = 3     # each repeat re-clusters with a different seed


def build_models():
    models = {
        "LogisticRegression": lambda: LogisticRegression(max_iter=2000, class_weight="balanced"),
        "RandomForest": lambda: RandomForestClassifier(
            n_estimators=300, min_samples_leaf=3, class_weight="balanced",
            random_state=42, n_jobs=-1),
        "MLP": lambda: MLPClassifier(hidden_layer_sizes=(32, 16), alpha=1e-3,
                                      max_iter=1500, random_state=42),
    }
    if HAS_XGB:
        models["XGBoost"] = lambda: XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42)
    return models


def clipped_log_loss(y_true, y_pred):
    y_pred = np.clip(y_pred, CLIP, 1 - CLIP)
    return log_loss(y_true, y_pred, labels=[0, 1])


def maybe_set_xgb_weight(model, y_train):
    if HAS_XGB and isinstance(model, XGBClassifier):
        pos, neg = max(y_train.sum(), 1), max(len(y_train) - y_train.sum(), 1)
        model.set_params(scale_pos_weight=neg / pos)


def fit_predict_single(train_fold, eval_fold, model_fn):
    """One model per species, predictions returned aligned to eval_fold rows."""
    preds = np.full(len(eval_fold), np.nan)
    for species, tr_sp in train_fold.groupby(SPECIES_COL):
        mask = (eval_fold[SPECIES_COL] == species).values
        ev_sp = eval_fold.loc[mask]
        if ev_sp.empty or tr_sp[TARGET].nunique() < 2:
            continue
        X_tr, y_tr = tr_sp[NUMERIC_FEATURES], tr_sp[TARGET].astype(int)
        X_ev = ev_sp[NUMERIC_FEATURES]
        model = model_fn()
        maybe_set_xgb_weight(model, y_tr)
        pipe = Pipeline([("scale", StandardScaler()), ("clf", model)])
        pipe.fit(X_tr, y_tr)
        preds[mask] = pipe.predict_proba(X_ev)[:, 1]
    return preds


def fit_predict_multi(train_fold, eval_fold, model_fn):
    """One pooled model, Species one-hot encoded additively."""
    X_tr = train_fold[NUMERIC_FEATURES + [SPECIES_COL]]
    y_tr = train_fold[TARGET].astype(int)
    X_ev = eval_fold[NUMERIC_FEATURES + [SPECIES_COL]]

    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), [SPECIES_COL]),
    ])
    model = model_fn()
    maybe_set_xgb_weight(model, y_tr)
    pipe = Pipeline([("prep", preprocessor), ("clf", model)])
    pipe.fit(X_tr, y_tr)
    return pipe.predict_proba(X_ev)[:, 1]


def evaluate_fold(train_fold, eval_fold, model_name, model_fn):
    """Returns overall + per-species log-loss for single and multi approaches
    on this one fold."""
    single_preds = fit_predict_single(train_fold, eval_fold, model_fn)
    multi_preds = fit_predict_multi(train_fold, eval_fold, model_fn)

    y_true = eval_fold[TARGET].astype(int).values
    valid = ~np.isnan(single_preds)  # species that had too little data to fit

    rows = []
    overall_single = clipped_log_loss(y_true[valid], single_preds[valid])
    overall_multi = clipped_log_loss(y_true, multi_preds)
    rows.append({"species": "__overall__",
                 "log_loss_single": overall_single, "log_loss_multi": overall_multi})

    for species in eval_fold[SPECIES_COL].unique():
        mask = (eval_fold[SPECIES_COL] == species).values & valid
        if mask.sum() == 0 or eval_fold.loc[eval_fold[SPECIES_COL] == species, TARGET].nunique() < 1:
            continue
        y_sp = eval_fold.loc[eval_fold[SPECIES_COL] == species, TARGET].astype(int).values
        multi_sp = multi_preds[(eval_fold[SPECIES_COL] == species).values]
        single_sp = single_preds[(eval_fold[SPECIES_COL] == species).values]
        if np.all(np.isnan(single_sp)):
            continue
        keep = ~np.isnan(single_sp)
        rows.append({
            "species": species,
            "log_loss_single": clipped_log_loss(y_sp[keep], single_sp[keep]),
            "log_loss_multi": clipped_log_loss(y_sp, multi_sp),
        })
    return pd.DataFrame(rows)


def random_group_cv_folds(data, n_splits, n_repeats, seed0=0):
    """Yields (train_fold, eval_fold) for repeated GroupKFold on `plot`."""
    plots = data[PLOT_COL].unique()
    for repeat in range(n_repeats):
        rng = np.random.RandomState(seed0 + repeat)
        shuffled_plots = rng.permutation(plots)
        # simple manual K-fold split over the shuffled plot list
        fold_assign = {p: i % n_splits for i, p in enumerate(shuffled_plots)}
        plot_fold = data[PLOT_COL].map(fold_assign)
        for fold in range(n_splits):
            eval_mask = (plot_fold == fold).values
            yield repeat, fold, data.loc[~eval_mask].copy(), data.loc[eval_mask].copy()


def spatial_block_cv_folds(data, n_blocks, n_repeats, seed0=100):
    """Yields (train_fold, eval_fold) using KMeans spatial blocks on plot
    centroids (easting/northing), leave-one-block-out, repeated with
    different cluster seeds so block boundaries vary."""
    plot_coords = data.groupby(PLOT_COL)[["easting", "northing"]].mean()
    for repeat in range(n_repeats):
        km = KMeans(n_clusters=n_blocks, random_state=seed0 + repeat, n_init=10)
        block_labels = km.fit_predict(plot_coords[["easting", "northing"]])
        plot_to_block = dict(zip(plot_coords.index, block_labels))
        block_of_row = data[PLOT_COL].map(plot_to_block)
        for block in range(n_blocks):
            eval_mask = (block_of_row == block).values
            if eval_mask.sum() == 0 or (~eval_mask).sum() == 0:
                continue
            yield repeat, block, data.loc[~eval_mask].copy(), data.loc[eval_mask].copy()


def run_cv_scheme(data, scheme_name, fold_generator, models):
    # IMPORTANT: materialise the folds ONCE so every model is evaluated on
    # the exact same folds (fair comparison), and so the generator isn't
    # silently exhausted after the first model (a generator can only be
    # iterated once).
    folds = list(fold_generator)
    all_rows = []
    for model_name, model_fn in models.items():
        print(f"  [{scheme_name}] {model_name} ...")
        for repeat, fold, train_fold, eval_fold in folds:
            fold_df = evaluate_fold(train_fold, eval_fold, model_name, model_fn)
            fold_df["model"] = model_name
            fold_df["repeat"] = repeat
            fold_df["fold"] = fold
            all_rows.append(fold_df)
    return pd.concat(all_rows, ignore_index=True)


def summarise(results_df, scheme_name, output_dir):
    summary = (
        results_df.groupby(["model", "species"])[["log_loss_single", "log_loss_multi"]]
        .agg(["mean", "std"])
    )
    summary.columns = ["_".join(c) for c in summary.columns]
    summary = summary.reset_index()
    summary["diff_mean (single - multi)"] = (
        summary["log_loss_single_mean"] - summary["log_loss_multi_mean"]
    )
    path = os.path.join(output_dir, f"{scheme_name}_summary.csv")
    summary.to_csv(path, index=False)
    print(f"[saved] {path}")

    # overall-only view, one bar per model, error bars = std across folds
    overall = summary[summary["species"] == "__overall__"]
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(overall))
    width = 0.35
    ax.bar(x - width / 2, overall["log_loss_single_mean"], width,
           yerr=overall["log_loss_single_std"], capsize=4, label="Single-species")
    ax.bar(x + width / 2, overall["log_loss_multi_mean"], width,
           yerr=overall["log_loss_multi_std"], capsize=4, label="Multi-species")
    ax.set_xticks(x)
    ax.set_xticklabels(overall["model"])
    ax.set_ylabel("Mean log-loss across folds (± std)")
    ax.set_title(f"{scheme_name.replace('_', ' ').title()}: overall log-loss with fold variability")
    ax.legend()
    fig.tight_layout()
    plot_path = os.path.join(output_dir, f"{scheme_name}_overall.png")
    fig.savefig(plot_path, dpi=150)
    plt.close(fig)
    print(f"[saved] {plot_path}")
    return summary


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    train = pd.read_csv(TRAIN_PATH)
    test = pd.read_csv(TEST_PATH)
    data = pd.concat([train, test], ignore_index=True)  # pool for CV — we
    # resample our own folds rather than relying on the one fixed split.

    models = build_models()

    print("Running repeated random (grouped-by-plot) CV ...")
    random_folds = random_group_cv_folds(data, N_SPLITS_RANDOM, N_REPEATS_RANDOM)
    random_results = run_cv_scheme(data, "random_cv", random_folds, models)
    random_summary = summarise(random_results, "random_cv", OUTPUT_DIR)

    print("\nRunning spatial block CV ...")
    spatial_folds = spatial_block_cv_folds(data, N_BLOCKS_SPATIAL, N_REPEATS_SPATIAL)
    spatial_results = run_cv_scheme(data, "spatial_cv", spatial_folds, models)
    spatial_summary = summarise(spatial_results, "spatial_cv", OUTPUT_DIR)

    print("\n===== RANDOM CV — overall (mean ± std across folds) =====")
    print(random_summary[random_summary["species"] == "__overall__"]
          [["model", "log_loss_single_mean", "log_loss_single_std",
            "log_loss_multi_mean", "log_loss_multi_std"]].to_string(index=False))

    print("\n===== SPATIAL CV — overall (mean ± std across folds) =====")
    print(spatial_summary[spatial_summary["species"] == "__overall__"]
          [["model", "log_loss_single_mean", "log_loss_single_std",
            "log_loss_multi_mean", "log_loss_multi_std"]].to_string(index=False))

    print("\nCompare the two tables above: if a model's winner flips between "
          "random and spatial CV, that's evidence random CV was overestimating "
          "generalisation to new locations (see Anticipated Issues in your plan).")


if __name__ == "__main__":
    main()

Running repeated random (grouped-by-plot) CV ...
  [random_cv] LogisticRegression ...
  [random_cv] RandomForest ...
  [random_cv] MLP ...


/opt/anaconda3/envs/macgpu/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/macgpu/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


  [random_cv] XGBoost ...
[saved] /Users/ak/Documents/UNSW Classes/DATA5925/CODE/single vs multi species/outputs/random_cv_summary.csv
[saved] /Users/ak/Documents/UNSW Classes/DATA5925/CODE/single vs multi species/outputs/random_cv_overall.png

Running spatial block CV ...
  [spatial_cv] LogisticRegression ...
  [spatial_cv] RandomForest ...
  [spatial_cv] MLP ...


/opt/anaconda3/envs/macgpu/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/macgpu/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/macgpu/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/macgpu/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/e

  [spatial_cv] XGBoost ...
[saved] /Users/ak/Documents/UNSW Classes/DATA5925/CODE/single vs multi species/outputs/spatial_cv_summary.csv
[saved] /Users/ak/Documents/UNSW Classes/DATA5925/CODE/single vs multi species/outputs/spatial_cv_overall.png

===== RANDOM CV — overall (mean ± std across folds) =====
             model  log_loss_single_mean  log_loss_single_std  log_loss_multi_mean  log_loss_multi_std
LogisticRegression              0.465862             0.029063             0.504306            0.020159
               MLP              0.384133             0.049541             0.259658            0.038687
      RandomForest              0.167713             0.015748             0.168542            0.011431
           XGBoost              0.190815             0.018991             0.253742            0.013669

===== SPATIAL CV — overall (mean ± std across folds) =====
             model  log_loss_single_mean  log_loss_single_std  log_loss_multi_mean  log_loss_multi_std
LogisticRegressi